# Module 16 lab · Research practice and the capstone

The code from every lesson of [this module](https://aicourses.dev/research/research-practice), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 16.2 Designing an experiment worth running

One-variable ablations, baselines, seeds and a written log of predictions: the judgement about which experiments are worth running, often called research taste.

[Read the lesson](https://aicourses.dev/research/research-practice/experiments-worth-running)

### Seeds and paired comparisons

Every run carries luck: the seed picks the starting weights and, here, the training data. To see how much, train a tiny network on a noisy XOR task, ten seeds, with and without a learning-rate warmup:

In [ ]:
import statistics as st
import torch
import torch.nn.functional as F

def xor_data(n, seed):                # XOR points, 10% labels flipped
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 2, generator=g)
    y = (X[:, 0] * X[:, 1] > 0).long()
    flip = torch.rand(n, generator=g) < 0.1
    return X, torch.where(flip, 1 - y, y)

def run(seed, warmup=1, steps=200):   # warmup=1 means no warmup
    X, y = xor_data(64, seed)          # the seed picks the training set
    Xv, yv = xor_data(2000, 999)       # one fixed validation set
    torch.manual_seed(seed)            # ...and the starting weights
    net = torch.nn.Sequential(torch.nn.Linear(2, 16), torch.nn.ReLU(),
                              torch.nn.Linear(16, 2))
    opt = torch.optim.AdamW(net.parameters(), lr=0.02)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: min(1.0, (t + 1) / warmup))
    for _ in range(steps):
        loss = F.cross_entropy(net(X), y)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    with torch.no_grad():
        return (net(Xv).argmax(1) == yv).float().mean().item()

`LambdaLR` multiplies the learning rate at step `t` by what its function returns: with `warmup=50`, a ramp from 1/50 up to 1 over the first 50 steps. Now compare the two arms seed by seed:

In [ ]:
base = [run(s) for s in range(10)]
warm = [run(s, warmup=50) for s in range(10)]
print("one lucky pair:", round(warm[3], 3), "vs", round(base[0], 3))
print("baseline range:", round(min(base), 3), "to", round(max(base), 3))
diffs = [w - b for w, b in zip(warm, base)]    # same seed, paired
se = st.stdev(diffs) / len(diffs) ** 0.5
print(f"paired gain: {st.mean(diffs):+.4f} ± {se:.4f}")
# one lucky pair: 0.806 vs 0.671
# baseline range: 0.671 to 0.837
# paired gain: +0.0036 ± 0.0039

## 16.4 Writing up results people finish

Claim, evidence, caveats and next step, with one headline figure that carries error bars.

[Read the lesson](https://aicourses.dev/research/research-practice/writing-results)

### One headline figure with error bars

The headline figure should answer the claim at a glance. For a paired experiment, plot each seed’s difference and the mean with its error bar, against a line at zero:

In [ ]:
import statistics as st
import matplotlib
matplotlib.use("Agg")                       # draw to a file, no window
import matplotlib.pyplot as plt

# lesson 16.2's accuracies, seeds 0 to 9, rounded
base = [0.671, 0.837, 0.709, 0.791, 0.788,
        0.796, 0.812, 0.805, 0.753, 0.752]
warm = [0.646, 0.836, 0.724, 0.806, 0.800,
        0.802, 0.805, 0.816, 0.762, 0.752]
diffs = [w - b for w, b in zip(warm, base)]
mean = st.mean(diffs)
se = st.stdev(diffs) / len(diffs) ** 0.5

fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.axhline(0, color="grey", lw=1)           # "no effect" line
ax.scatter(range(10), diffs, color="tab:blue", label="one seed")
ax.errorbar(11, mean, yerr=2 * se, fmt="o", color="black",
            capsize=4, label="mean ± 2 SE")
ax.set_xlabel("seed")
ax.set_ylabel("accuracy with warmup\nminus without")
ax.set_title("Warmup: no detectable effect (10 paired seeds)")
ax.legend(frameon=False)
fig.savefig("headline.png", dpi=150)
print(f"{mean:+.4f} ± {se:.4f}")            # +0.0035 ± 0.0039

## 16.5 Reading a big ML codebase and landing a PR

A merged, non-trivial PR to a major ML library is public proof that you can work in a large shared codebase, which is much of a research engineer’s daily work.

[Read the lesson](https://aicourses.dev/research/research-practice/contributing-upstream)

### Learn the route, not the map

Start from something you already call, such as `model.generate()`, and find where it lives. Python can tell you without opening any file, with `inspect`, its built-in module for examining code:

In [ ]:
import inspect
from transformers import GPT2LMHeadModel

def where(obj):              # the file and line that define obj
    obj = inspect.unwrap(obj)    # look through decorators
    file = inspect.getsourcefile(obj)
    line = inspect.getsourcelines(obj)[1]
    return file.split("transformers/")[-1], line

print(where(GPT2LMHeadModel))
print(where(GPT2LMHeadModel.forward))
print(where(GPT2LMHeadModel.generate))